# Generic Checkpoint Eval With Mamba Generation

This notebook does not train. It loads any saved LoRA checkpoint adapter from Drive, can run the fixed probe/eval split from a trace CSV, writes raw generated completions to Drive, packages a Kaggle `submission.zip`, and creates a local archive bundle for dashboard/history.

Recommended order:

1. Run runtime/config/data-loading cells.
2. Run the artifact audit cell before loading/generating if you want to see what already exists.
3. Run only the missing parts you want: lightweight run-config/trainer-log, probe, generated eval, submission zip, sanity test, archive bundle.
4. Download `submission.zip` for Kaggle and the archive bundle for local dashboard/history.

Use it while another Colab runtime is training or resuming. Configure only the top configuration cell.

Current exp11 checkpoint sweep: run checkpoints `195`, `390`, `585`, and `780` one at a time by changing only `CHECKPOINT_STEP`, then download each archive bundle. Use the same `EVAL_ROWS`, prompt, max tokens, and generation batch size for all four checkpoints so the summaries are comparable.


In [ ]:
# Runtime and installs. Run first in a fresh GPU Colab runtime.
import torch
print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("Use a GPU runtime.")
print("gpu:", torch.cuda.get_device_name(0))
print("bf16 supported:", torch.cuda.is_bf16_supported())
if not torch.cuda.is_bf16_supported():
    raise RuntimeError("Use an A100/H100 or another BF16-capable runtime.")

# Fused Mamba is used for generation speed only. This notebook never trains.
!pip install -q packaging ninja bitsandbytes accelerate peft transformers safetensors
!pip install -q mamba-ssm causal-conv1d --no-build-isolation

import importlib.util
print("mamba_ssm available:", importlib.util.find_spec("mamba_ssm") is not None)
print("causal_conv1d available:", importlib.util.find_spec("causal_conv1d") is not None)

from google.colab import drive
drive.mount("/content/drive", force_remount=False)


In [ ]:
# Configuration.
from pathlib import Path

from google.colab import files

# Change these for each eval run.
EXPERIMENT_NAME = "exp11_mamba_trace_v2_aug25k_b8_ep1"
CHECKPOINT_STEP = 195  # exp11 checkpoints to sweep: 195, 390, 585, 780
EVAL_RUN_LABEL = None  # None -> f"{EXPERIMENT_NAME}_checkpoint-{CHECKPOINT_STEP}_mamba"

# Data/model settings should usually stay aligned with the training notebook.
MODEL_NAME = "nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-BF16"
TRACE_TRAINING_CSV_PATH = Path("/content/trace_v2_aug25k.csv")
TEST_CSV_PATH = Path("/content/test.csv")
SYSTEM_PROMPT = r"Solve the puzzle. Use concise reasoning when helpful. End with exactly one final answer inside \boxed{}."
EVAL_ROWS = 64
PROBE_ROWS = 5
EVAL_RANDOM_STATE = 42
MAX_SEQ_LENGTH = 512
MAX_NEW_TOKENS = 384
GENERATION_BATCH_SIZE = 12
PRECISION_MODE = "bf16"

# Fused Mamba gives faster no-cache generation. Manual cache was rejected because it changed outputs.
INSTALL_FUSED_MAMBA = True

DRIVE_ARTEFACTS_ROOT = Path("/content/drive/MyDrive/Colab_Notebooks/Kaggle challenges/nemotron_challenge/artefacts")
TRAIN_OUTPUT_DIR = DRIVE_ARTEFACTS_ROOT / "outputs" / EXPERIMENT_NAME
CHECKPOINT_DIR = TRAIN_OUTPUT_DIR / f"checkpoint-{CHECKPOINT_STEP}"
BACKFILL_OUTPUT_DIR = DRIVE_ARTEFACTS_ROOT / "eval_backfill" / "outputs"
OUTPUT_LABEL = EVAL_RUN_LABEL or f"{EXPERIMENT_NAME}_checkpoint-{CHECKPOINT_STEP}_mamba"
OUTPUT_DIR = BACKFILL_OUTPUT_DIR / OUTPUT_LABEL
TRAINER_LOG_SOURCE_PATH = TRAIN_OUTPUT_DIR / "trainer_log_history.csv"
TRAINER_STATE_PATH = CHECKPOINT_DIR / "trainer_state.json"
TRAINER_LOG_CSV_PATH = OUTPUT_DIR / "trainer_log_history.csv"

if not CHECKPOINT_DIR.is_dir():
    raise FileNotFoundError(f"checkpoint not found: {CHECKPOINT_DIR}")
if not TRACE_TRAINING_CSV_PATH.exists():
    raise FileNotFoundError(f"Upload the trace CSV to {TRACE_TRAINING_CSV_PATH}")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("experiment:", EXPERIMENT_NAME)
print("checkpoint step:", CHECKPOINT_STEP)
print("checkpoint:", CHECKPOINT_DIR)
print("output label:", OUTPUT_LABEL)
print("output:", OUTPUT_DIR)


In [ ]:
from __future__ import annotations

import json
import math
import re
import time
import types
import zipfile
from pathlib import Path

import pandas as pd


def infer_family(prompt: str) -> str:
    text = str(prompt).lower()
    if "bit manipulation" in text or "8-bit binary" in text:
        return "bit_manipulation"
    if "secret encryption" in text or "decrypt" in text:
        return "cipher"
    if "numeral system" in text:
        return "numeral"
    if "unit" in text and "convert" in text:
        return "unit_conversion"
    if "gravitational constant" in text or "falling distance" in text:
        return "gravity"
    if "transformation rules is applied to equations" in text or "determine the result for" in text:
        return "equation_symbolic"
    return "unknown"


def normalize_answer(value) -> str:
    text = "" if value is None else str(value).strip()
    text = text.strip(chr(96))
    text = re.sub(r"\s+", " ", text)
    return text.strip().strip(".")


def extract_final_answer(text: str | None) -> str:
    if text is None:
        return "NOT_FOUND"
    boxed_starts = list(re.finditer(r"\\boxed\{", text))
    matches = []
    for idx, match in enumerate(boxed_starts):
        start = match.end()
        end = boxed_starts[idx + 1].start() if idx + 1 < len(boxed_starts) else len(text)
        segment = text[start:end]
        last_brace = segment.rfind("}")
        matches.append(segment[:last_brace] if last_brace != -1 else segment)
    if matches:
        non_empty = [match.strip() for match in matches if match.strip()]
        if non_empty:
            return non_empty[-1]
        return matches[-1].strip()

    patterns = [
        r"The final answer is:\s*([^\n]+)",
        r"Final answer is:\s*([^\n]+)",
        r"Final answer\s*[:\uFF1A]\s*([^\n]+)",
        r"final answer\s*[:\uFF1A]\s*([^\n]+)",
    ]
    for pattern in patterns:
        found = re.findall(pattern, text, re.IGNORECASE)
        if found:
            return found[-1].strip()
    number_matches = re.findall(r"-?\d+(?:\.\d+)?", text)
    if number_matches:
        return number_matches[-1]
    lines = [line.strip() for line in text.splitlines() if line.strip()]
    return lines[-1] if lines else "NOT_FOUND"


def verify_answer(stored_answer: str, predicted: str) -> bool:
    stored_answer = str(stored_answer).strip()
    predicted = str(predicted).strip()
    if re.fullmatch(r"[01]+", stored_answer):
        return predicted.lower() == stored_answer.lower()
    try:
        return math.isclose(float(stored_answer), float(predicted), rel_tol=1e-2, abs_tol=1e-5)
    except Exception:
        return predicted.lower() == stored_answer.lower()


def build_prompt(question: str) -> str:
    return f"System:\n{SYSTEM_PROMPT}\n\nUser:\n{question}\n\nAssistant:\n"


def load_rows() -> tuple[pd.DataFrame, pd.DataFrame]:
    train = pd.read_csv(TRACE_TRAINING_CSV_PATH, dtype=str).fillna("")
    required = {"id", "question", "trace", "gold_answer"}
    missing = required - set(train.columns)
    if missing:
        raise ValueError(f"trace_training.csv missing columns: {sorted(missing)}")
    rows = train[["id", "question", "gold_answer"]].copy()
    rows["gold_answer"] = rows["gold_answer"].map(normalize_answer)
    rows["family"] = rows["question"].map(infer_family)
    eval_rows = rows.sample(n=min(EVAL_ROWS, len(rows) - 1), random_state=EVAL_RANDOM_STATE).reset_index(drop=True)
    probe_rows = rows.groupby("family", group_keys=False).head(1).head(PROBE_ROWS).reset_index(drop=True)
    return eval_rows, probe_rows


eval_rows, probe_rows = load_rows()
print("eval rows:", len(eval_rows))
print("probe rows:", len(probe_rows))
display(probe_rows[["id", "family", "gold_answer"]])

In [ ]:
# Audit which checkpoint backfill artifacts exist and which still need to be generated.
EXPECTED_OUTPUT_FILES = {
    "kaggle_submission_zip": OUTPUT_DIR / "submission.zip",
    "run_config": OUTPUT_DIR / "run_config.json",
    "trainer_log_history": OUTPUT_DIR / "trainer_log_history.csv",
    "probe_evolution": OUTPUT_DIR / "probe_evolution.csv",
    "generated_eval_summary": OUTPUT_DIR / "generated_eval_summary.csv",
    "generated_eval_predictions": OUTPUT_DIR / "generated_eval_predictions.csv",
    "sanity_test_predictions": OUTPUT_DIR / "sanity_test_predictions.csv",
    "sanity_test_predictions_raw": OUTPUT_DIR / "sanity_test_predictions_raw.csv",
    "archive_manifest": OUTPUT_DIR / "archive_manifest.json",
    "archive_bundle": OUTPUT_DIR / f"{OUTPUT_LABEL}_archive_bundle.zip",
}


def audit_backfill_outputs() -> pd.DataFrame:
    rows = []
    for name, path in EXPECTED_OUTPUT_FILES.items():
        exists = path.exists()
        if name.startswith("sanity_test") and not TEST_CSV_PATH.exists():
            status = "optional: /content/test.csv missing"
            action = "upload /content/test.csv, then rerun sanity cell if wanted"
        elif exists:
            status = "ok"
            action = "none"
        else:
            status = "missing"
            if name in {"run_config", "trainer_log_history"}:
                action = "run lightweight run-config/trainer-log cell"
            elif name in {"generated_eval_summary", "generated_eval_predictions"}:
                action = "rerun generated-eval cell"
            elif name == "probe_evolution":
                action = "rerun probe cell"
            elif name == "kaggle_submission_zip":
                action = "rerun submission zip cell"
            elif name.startswith("sanity_test"):
                action = "rerun sanity cell"
            elif name in {"archive_manifest", "archive_bundle"}:
                action = "rerun archive bundle cell after missing files are generated"
            else:
                action = "check notebook"
        rows.append({
            "artifact": name,
            "status": status,
            "action": action,
            "path": str(path),
        })
    audit = pd.DataFrame(rows)
    display(audit)
    missing = audit[audit["status"].astype(str).str.startswith("missing")]
    if len(missing):
        print("Missing required/current artifacts:", ", ".join(missing["artifact"].tolist()))
    else:
        print("All current artifacts are present. Optional sanity files depend on /content/test.csv.")
    return audit


backfill_audit = audit_backfill_outputs()


In [ ]:
# Lightweight metadata/log write. Run this if audit says run_config or trainer_log_history is missing.
def write_trainer_log_history() -> None:
    if TRAINER_LOG_SOURCE_PATH.exists():
        trainer_log = pd.read_csv(TRAINER_LOG_SOURCE_PATH)
        trainer_log.to_csv(TRAINER_LOG_CSV_PATH, index=False)
        print("wrote trainer log from training output:", TRAINER_LOG_CSV_PATH)
        return

    if TRAINER_STATE_PATH.exists():
        trainer_state = json.loads(TRAINER_STATE_PATH.read_text(encoding="utf-8"))
        log_history = trainer_state.get("log_history", [])
        if log_history:
            pd.DataFrame(log_history).to_csv(TRAINER_LOG_CSV_PATH, index=False)
            print("wrote trainer log from checkpoint trainer_state.json:", TRAINER_LOG_CSV_PATH)
            return

    print("trainer log not found; checked:", TRAINER_LOG_SOURCE_PATH, "and", TRAINER_STATE_PATH)


def write_run_config() -> None:
    run_config_path = OUTPUT_DIR / "run_config.json"
    run_config = {
        "experiment_name": EXPERIMENT_NAME,
        "checkpoint_step": CHECKPOINT_STEP,
        "checkpoint_dir": str(CHECKPOINT_DIR),
        "output_dir": str(OUTPUT_DIR),
        "trainer_log_source_path": str(TRAINER_LOG_SOURCE_PATH),
        "trainer_state_path": str(TRAINER_STATE_PATH),
        "trainer_log_csv_path": str(TRAINER_LOG_CSV_PATH),
        "trace_training_csv_path": str(TRACE_TRAINING_CSV_PATH),
        "test_csv_path": str(TEST_CSV_PATH),
        "system_prompt": SYSTEM_PROMPT,
        "eval_rows": len(eval_rows),
        "probe_rows": len(probe_rows),
        "max_seq_length": MAX_SEQ_LENGTH,
        "max_new_tokens": MAX_NEW_TOKENS,
        "generation_batch_size": GENERATION_BATCH_SIZE,
        "generation_use_cache": False,
        "mamba_backend": "fused causal-conv1d installed for generation; cache disabled by default",
        "eval_random_state": EVAL_RANDOM_STATE,
        "output_label": OUTPUT_LABEL,
    }
    run_config_path.write_text(json.dumps(run_config, indent=2), encoding="utf-8")
    print("wrote", run_config_path)


write_trainer_log_history()
write_run_config()


In [ ]:
def patch_nemotron_moe_dtype(current_model, torch_module) -> None:
    def _patched_nemotron_moe(self, hidden_states, topk_indices, topk_weights):
        final_hidden_states = torch_module.zeros_like(hidden_states, dtype=topk_weights.dtype)
        expert_mask = torch_module.nn.functional.one_hot(topk_indices, num_classes=len(self.experts))
        expert_mask = expert_mask.permute(2, 0, 1)
        for expert_idx in range(len(self.experts)):
            expert = self.experts[expert_idx]
            mask = expert_mask[expert_idx]
            token_indices, weight_indices = torch_module.where(mask)
            if token_indices.numel() > 0:
                expert_weights = topk_weights[token_indices, weight_indices]
                expert_input = hidden_states[token_indices]
                expert_output = expert(expert_input)
                weighted_output = expert_output * expert_weights.unsqueeze(-1)
                final_hidden_states.index_add_(0, token_indices, weighted_output.to(final_hidden_states.dtype))
            else:
                expert_dtype = expert.down_proj.weight.dtype
                dummy_input = torch_module.zeros_like(hidden_states[0]).unsqueeze(0).to(expert_dtype)
                final_hidden_states = final_hidden_states + expert(dummy_input).to(final_hidden_states.dtype)
        return final_hidden_states.to(hidden_states.dtype)

    patched = 0
    for module in current_model.modules():
        if module.__class__.__name__ == "NemotronHMOE":
            module.moe = types.MethodType(_patched_nemotron_moe, module)
            patched += 1
    print("patched Nemotron MoE dtype modules:", patched)


def map_adapter_key(key: str) -> str:
    mapped = key
    for suffix in ["lora_A", "lora_B", "lora_embedding_A", "lora_embedding_B"]:
        mapped = mapped.replace(f".{suffix}.weight", f".{suffix}.default.weight")
    return mapped


def load_adapter_weights_directly(model, adapter_dir: Path) -> None:
    from safetensors.torch import load_file

    adapter_path = adapter_dir / "adapter_model.safetensors"
    adapter_state = load_file(str(adapter_path))
    model_state = model.state_dict()
    mapped_state = {}
    missing = []
    for key, tensor in adapter_state.items():
        mapped_key = map_adapter_key(key)
        candidates = [mapped_key]
        if not mapped_key.startswith("base_model.model."):
            candidates.append("base_model.model." + mapped_key)
        target_key = next((candidate for candidate in candidates if candidate in model_state), None)
        if target_key is None:
            missing.append(key)
        else:
            mapped_state[target_key] = tensor.to(model_state[target_key].dtype)

    if missing:
        preview = "\n".join(missing[:20])
        raise KeyError(f"Could not map {len(missing)} adapter weights. First missing keys:\n{preview}")
    result = model.load_state_dict(mapped_state, strict=False)
    print(f"loaded {len(mapped_state)} adapter tensors directly from {adapter_path}")
    if result.unexpected_keys:
        print("unexpected keys:", result.unexpected_keys[:10])


def load_model(adapter_dir: Path):
    import torch
    from google.colab import userdata
    from peft import LoraConfig, get_peft_model
    from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

    adapter_config = json.loads((adapter_dir / "adapter_config.json").read_text(encoding="utf-8"))
    model_name = adapter_config.get("base_model_name_or_path") or MODEL_NAME
    hf_token = userdata.get("hftoken")

    tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True, token=hf_token)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.bfloat16,
    )
    base_model = AutoModelForCausalLM.from_pretrained(
        model_name,
        quantization_config=bnb_config,
        torch_dtype=torch.bfloat16,
        device_map={"": 0},
        trust_remote_code=True,
        token=hf_token,
    )
    base_model.config.use_cache = False
    base_model.generation_config.use_cache = False
    patch_nemotron_moe_dtype(base_model, torch)

    lora_config = LoraConfig(
        task_type=adapter_config.get("task_type", "CAUSAL_LM"),
        inference_mode=True,
        r=int(adapter_config["r"]),
        lora_alpha=int(adapter_config["lora_alpha"]),
        lora_dropout=float(adapter_config.get("lora_dropout", 0.0)),
        target_modules=adapter_config["target_modules"],
        bias=adapter_config.get("bias", "none"),
    )
    model = get_peft_model(base_model, lora_config)
    load_adapter_weights_directly(model, adapter_dir)
    model.eval()
    return model, tokenizer, torch


model, tokenizer, torch_module = load_model(CHECKPOINT_DIR)

In [ ]:
def generate_answers(model, tokenizer, torch_module, rows: pd.DataFrame, phase: str, step: int, max_new_tokens: int) -> pd.DataFrame:
    records = []
    was_training = model.training
    model.eval()
    device = next(model.parameters()).device
    batch_size = max(1, int(GENERATION_BATCH_SIZE))
    previous_padding_side = tokenizer.padding_side
    tokenizer.padding_side = "left"
    started = time.time()
    print(f"{phase}: generating {len(rows)} rows in batches of {batch_size}", flush=True)
    try:
        for start_index in range(0, len(rows), batch_size):
            batch_rows = rows.iloc[start_index:start_index + batch_size].reset_index(drop=True)
            prompts = [build_prompt(question) for question in batch_rows["question"]]
            inputs = tokenizer(
                prompts,
                return_tensors="pt",
                truncation=True,
                max_length=MAX_SEQ_LENGTH,
                padding=True,
            ).to(device)
            prompt_width = inputs["input_ids"].shape[1]
            start = time.time()
            with torch_module.no_grad():
                output_ids = model.generate(
                    **inputs,
                    max_new_tokens=max_new_tokens,
                    do_sample=False,
                    use_cache=False,
                    pad_token_id=tokenizer.eos_token_id,
                    eos_token_id=tokenizer.eos_token_id,
                )
            batch_seconds = time.time() - start
            seconds_per_row = batch_seconds / max(1, len(batch_rows))
            for local_idx, row in enumerate(batch_rows.itertuples(index=False)):
                generated_ids = output_ids[local_idx, prompt_width:]
                eos_positions = (generated_ids == tokenizer.eos_token_id).nonzero(as_tuple=True)[0]
                has_eos = len(eos_positions) > 0
                if has_eos:
                    generated_ids = generated_ids[: int(eos_positions[0]) + 1]
                raw = tokenizer.decode(generated_ids, skip_special_tokens=True).strip()
                answer = extract_final_answer(raw)
                records.append({
                    "phase": phase,
                    "step": step,
                    "id": row.id,
                    "family": row.family,
                    "gold": row.gold_answer,
                    "answer": answer,
                    "match": verify_answer(row.gold_answer, answer),
                    "seconds": seconds_per_row,
                    "generated_tokens": int(generated_ids.numel()),
                    "hit_max_new_tokens": (not has_eos) and int(generated_ids.numel()) >= max_new_tokens,
                    "raw_output": raw,
                })
            done = min(start_index + len(batch_rows), len(rows))
            elapsed = time.time() - started
            print(f"{phase}: {done}/{len(rows)} rows, elapsed={elapsed:.1f}s, last_batch={batch_seconds:.1f}s", flush=True)
    finally:
        tokenizer.padding_side = previous_padding_side
    if was_training:
        model.train()
    scored = pd.DataFrame(records)
    scored["empty_answer"] = scored["answer"].astype(str).str.len().eq(0)
    return scored


def summarize(scored: pd.DataFrame, checkpoint: str) -> pd.DataFrame:
    rows = []
    for family, part in [("all", scored)] + list(scored.groupby("family", dropna=False)):
        rows.append({
            "checkpoint": checkpoint,
            "step": CHECKPOINT_STEP,
            "family": family,
            "rows": int(len(part)),
            "matches": int(part["match"].sum()),
            "accuracy": float(part["match"].mean()) if len(part) else 0.0,
            "empty_answers": int(part["empty_answer"].sum()),
            "hit_max_new_tokens": int(part["hit_max_new_tokens"].sum()),
            "avg_generated_tokens": float(part["generated_tokens"].mean()) if len(part) else 0.0,
            "avg_seconds": float(part["seconds"].mean()) if len(part) else 0.0,
        })
    return pd.DataFrame(rows)


In [ ]:
# Run probe generation only if audit says probe_evolution is missing or stale.
probe_scored = generate_answers(model, tokenizer, torch_module, probe_rows, "probe_checkpoint", CHECKPOINT_STEP, MAX_NEW_TOKENS)
probe_path = OUTPUT_DIR / "probe_evolution.csv"
probe_scored.to_csv(probe_path, index=False)
print("wrote", probe_path)
display(probe_scored[["phase", "step", "id", "family", "gold", "answer", "match", "generated_tokens", "hit_max_new_tokens", "raw_output"]])

In [ ]:
# Run generated eval only if audit says generated_eval files are missing or stale.
eval_scored = generate_answers(model, tokenizer, torch_module, eval_rows, "generated_eval_checkpoint", CHECKPOINT_STEP, MAX_NEW_TOKENS)
summary = summarize(eval_scored, f"checkpoint-{CHECKPOINT_STEP}")

predictions_path = OUTPUT_DIR / "generated_eval_predictions.csv"
summary_path = OUTPUT_DIR / "generated_eval_summary.csv"
run_config_path = OUTPUT_DIR / "run_config.json"

eval_scored.to_csv(predictions_path, index=False)
summary.to_csv(summary_path, index=False)


print("wrote", predictions_path)
print("wrote", summary_path)
display(summary)

In [ ]:
# Package this checkpoint as a Kaggle submission adapter zip.
submission_zip_path = OUTPUT_DIR / "submission.zip"
required_adapter_files = ["adapter_config.json", "adapter_model.safetensors"]

for name in required_adapter_files:
    path = CHECKPOINT_DIR / name
    if not path.exists():
        raise FileNotFoundError(f"missing required adapter file: {path}")

adapter_config = json.loads((CHECKPOINT_DIR / "adapter_config.json").read_text(encoding="utf-8"))
rank = adapter_config.get("r")
if rank is not None and int(rank) > 32:
    raise ValueError(f"Kaggle requires LoRA rank <= 32, got r={rank}")

with zipfile.ZipFile(submission_zip_path, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for name in required_adapter_files:
        archive.write(CHECKPOINT_DIR / name, arcname=name)

with zipfile.ZipFile(submission_zip_path, "r") as archive:
    names = sorted(archive.namelist())

expected = sorted(required_adapter_files)
if names != expected:
    raise AssertionError(f"submission.zip has wrong files: {names}")

print("wrote Kaggle submission zip:", submission_zip_path)
print("zip contents:", names)
print("adapter rank:", rank)


In [ ]:
# Kaggle does not use these predictions; upload only submission.zip.
# Run this only if audit says sanity files are missing and /content/test.csv exists.
test_rows = pd.read_csv(TEST_CSV_PATH, dtype=str).fillna("")
test_rows["gold_answer"] = ""
test_rows["family"] = test_rows["question"].map(infer_family)

sanity_scored = generate_answers(
    model,
    tokenizer,
    torch_module,
    test_rows[["id", "question", "gold_answer", "family"]],
    phase="test_sanity",
    step=CHECKPOINT_STEP,
    max_new_tokens=MAX_NEW_TOKENS,
)

sanity_raw_path = OUTPUT_DIR / "sanity_test_predictions_raw.csv"
sanity_csv_path = OUTPUT_DIR / "sanity_test_predictions.csv"
sanity_scored.to_csv(sanity_raw_path, index=False)
sanity_scored[["id", "answer"]].to_csv(sanity_csv_path, index=False)

print("wrote", sanity_raw_path)
print("wrote", sanity_csv_path)
display(sanity_scored[["id", "family", "answer", "generated_tokens", "hit_max_new_tokens", "raw_output"]])



In [ ]:
# Build and download a dashboard/archive bundle.
# This is for local project tracking. It is not the Kaggle upload artifact.
archive_bundle_path = OUTPUT_DIR / f"{OUTPUT_LABEL}_archive_bundle.zip"

wanted_files = [
    OUTPUT_DIR / "submission.zip",
    OUTPUT_DIR / "run_config.json",
    OUTPUT_DIR / "probe_evolution.csv",
    OUTPUT_DIR / "trainer_log_history.csv",
    OUTPUT_DIR / "generated_eval_summary.csv",
    OUTPUT_DIR / "generated_eval_predictions.csv",
    OUTPUT_DIR / "sanity_test_predictions.csv",
    OUTPUT_DIR / "sanity_test_predictions_raw.csv",
]

manifest = {
    "experiment_name": EXPERIMENT_NAME,
    "checkpoint_step": CHECKPOINT_STEP,
    "output_label": OUTPUT_LABEL,
    "output_dir": str(OUTPUT_DIR),
    "checkpoint_dir": str(CHECKPOINT_DIR),
    "included": [],
    "missing": [],
}

with zipfile.ZipFile(archive_bundle_path, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for path in wanted_files:
        if path.exists():
            archive.write(path, arcname=path.name)
            manifest["included"].append(path.name)
        else:
            manifest["missing"].append(path.name)

    manifest_path = OUTPUT_DIR / "archive_manifest.json"
    manifest_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")
    archive.write(manifest_path, arcname="archive_manifest.json")

print("archive bundle:", archive_bundle_path)
print("included:", manifest["included"])
print("missing:", manifest["missing"])
files.download(str(archive_bundle_path))


In [ ]:
# Download the Kaggle upload artifact only.
files.download(str(OUTPUT_DIR / "submission.zip"))


In [ ]:
backfill_audit = audit_backfill_outputs()
